# Save Best Models (.pkl / .pth)

Saves best model per dataset. Run notebooks **01** and **02** first, or run the setup cell below.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
DOCS = ROOT / 'docs'
MODELS = ROOT / 'models'
for p in (RESULTS, DOCS, MODELS):
    p.mkdir(exist_ok=True)

DATA_DIR = ROOT / 'data'
for candidate in [ROOT / 'data', ROOT, ROOT.parent,
                  Path(r'C:\kaggle\input\cmapss-jet-engine-simulated-data'),
                  Path('/kaggle/input/cmapss-jet-engine-simulated-data')]:
    if candidate.is_dir() and list(candidate.glob('train_FD*.txt')):
        DATA_DIR = candidate
        break
print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)

In [ ]:
# Re-use ML helpers (run notebook 01 first, or define here)
import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')
RUL_CAP, ROLLING_WINDOW = 125, 5
DATASET_IDS = ['FD001','FD002','FD003','FD004']
COL_NAMES = ['unit','cycle','op_setting_1','op_setting_2','op_setting_3'] + [f'sensor_{i+1}' for i in range(21)]

def discover_files(data_dir):
    mapping = {}
    for path in glob.glob(os.path.join(str(data_dir), '*')):
        name = os.path.basename(path).upper()
        for fd in DATASET_IDS:
            if fd in name:
                mapping.setdefault(fd, {})
                if 'TRAIN' in name: mapping[fd]['train'] = path
                if 'TEST' in name: mapping[fd]['test'] = path
                if 'RUL' in name: mapping[fd]['rul'] = path
    return mapping

def load_dataset(train_path, test_path, rul_path):
    train = pd.read_csv(train_path, sep=r'\s+', header=None, names=COL_NAMES)
    train['RUL'] = train.groupby('unit')['cycle'].transform('max') - train['cycle']
    test = pd.read_csv(test_path, sep=r'\s+', header=None, names=COL_NAMES)
    rul = pd.read_csv(rul_path, sep=r'\s+', header=None, names=['RUL_final'])
    max_cycle = test.groupby('unit')['cycle'].max().reset_index().sort_values('unit')
    max_cycle['RUL_final'] = rul['RUL_final'].values
    test = test.merge(max_cycle[['unit','RUL_final']], on='unit', how='left')
    test['RUL'] = test.groupby('unit')['cycle'].transform('max') - test['cycle'] + test['RUL_final']
    return train, test.drop(columns=['RUL_final'])

def select_features(train_df, cols, min_std=1e-6):
    stds = train_df[cols].std()
    return stds[stds > min_std].index.tolist()

def build_feature_matrix(df, base_cols, window=5):
    parts, grouped = [], df.groupby('unit', sort=False)
    for col in base_cols:
        parts += [df[[col]],
            grouped[col].transform(lambda s: s-s.iloc[0]).to_frame(f'{col}_rel'),
            grouped[col].transform(lambda s: s.rolling(window,min_periods=1).mean()).to_frame(f'{col}_rmean'),
            grouped[col].transform(lambda s: s.rolling(window,min_periods=1).std().fillna(0)).to_frame(f'{col}_rstd')]
    return pd.concat(parts, axis=1)

def prepare_xy(train_df, test_df, feature_cols):
    train_feat = build_feature_matrix(train_df, feature_cols)
    test_feat = build_feature_matrix(test_df, feature_cols)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_feat)
    X_test = scaler.transform(test_feat)
    return X_train, train_df['RUL'].clip(upper=RUL_CAP).values, X_test, test_df['RUL'].values, list(train_feat.columns)

file_map = discover_files(DATA_DIR)

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
import xgboost as xgb

def get_models(n_train):
    large = n_train > 40000
    n_est = 120 if large else 250
    return {
        'ExtraTrees': ExtraTreesRegressor(n_estimators=n_est, max_depth=24, n_jobs=-1, random_state=42),
        'HistGradientBoosting': HistGradientBoostingRegressor(
            max_iter=250 if large else 400, learning_rate=0.06, max_depth=12, random_state=42),
    }

## Configuration — best models from benchmark

In [ ]:
import json, pickle, joblib
import torch

BEST = {
    'FD001': {'family': 'ML', 'model': 'ExtraTrees'},
    'FD002': {'family': 'DL', 'model': 'Transformer'},
    'FD003': {'family': 'ML', 'model': 'HistGradientBoosting'},
    'FD004': {'family': 'ML', 'model': 'HistGradientBoosting'},
}

## Save ML models

In [ ]:
# Requires helpers from notebook 01 (discover_files, load_dataset, select_features, prepare_xy, get_models, build_feature_matrix)
from sklearn.preprocessing import StandardScaler

manifest = []
file_map = discover_files(DATA_DIR)

for fd, spec in BEST.items():
    if spec['family'] != 'ML':
        continue
    train_df, test_df = load_dataset(file_map[fd]['train'], file_map[fd]['test'], file_map[fd]['rul'])
    candidate = [c for c in train_df.columns if c.startswith('sensor_') or c.startswith('op_setting_')]
    feature_cols = select_features(train_df, candidate)
    X_train, y_train, X_test, y_test, expanded_cols = prepare_xy(train_df, test_df, feature_cols)
    train_feat = build_feature_matrix(train_df, feature_cols)
    scaler = StandardScaler()
    scaler.fit(train_feat)
    model = get_models(len(X_train))[spec['model']]
    model.fit(X_train, y_train)
    bundle = {'model': model, 'scaler': scaler, 'feature_cols': feature_cols,
              'expanded_cols': expanded_cols, 'dataset': fd, 'model_name': spec['model'],
              'family': 'ML', 'rul_cap': 125}
    out = MODELS / f"{fd}_{spec['model']}.pkl"
    joblib.dump(bundle, out)
    manifest.append({'dataset': fd, 'model': spec['model'], 'family': 'ML', 'path': str(out)})
    print('Saved', out)

## Save DL model (Transformer FD002)

> Run **notebook 02** first so `build_model`, `prepare_sequence_data`, `train_model`, `SEQ_LEN`, and `DEVICE` are defined.

In [ ]:
# Requires notebook 02 definitions (build_model, prepare_sequence_data, train_model, DEVICE, SEQ_LEN)
fd, model_name = 'FD002', 'Transformer'
train_df, test_df = load_dataset(file_map[fd]['train'], file_map[fd]['test'], file_map[fd]['rul'])
candidate = [c for c in train_df.columns if c.startswith('sensor_') or c.startswith('op_setting_')]
feature_cols = select_features(train_df, candidate)
X_train, y_train, train_meta, X_test, y_test, test_meta = prepare_sequence_data(
    train_df, test_df, feature_cols, SEQ_LEN)
batch = 16
model = build_model(model_name, X_train.shape[2])
model = train_model(model, X_train, y_train, train_meta, batch_size=batch)
meta = {'dataset': fd, 'model_name': model_name, 'family': 'DL',
        'feature_cols': feature_cols, 'seq_len': SEQ_LEN, 'n_features': X_train.shape[2]}
pth = MODELS / f'{fd}_{model_name}.pth'
pkl = MODELS / f'{fd}_{model_name}_meta.pkl'
torch.save(model.state_dict(), pth)
with open(pkl, 'wb') as f:
    pickle.dump(meta, f)
manifest.append({'dataset': fd, 'model': model_name, 'family': 'DL',
                 'weights': str(pth), 'metadata': str(pkl)})
print('Saved', pth, pkl)

## Write manifest

In [ ]:
(MODELS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))